In [19]:
import pandas as pd
pd.set_option('display.max_columns', 500)
import numpy as np
import mne
import os
from mne_bids import BIDSPath, read_raw_bids
from pathlib import Path
import torch
from tqdm import tqdm

In [ ]:
EOEC_means = pd.read_csv('/home/azureuser/mycontainer/HBN-Processed/EOEC_means.csv')
EOEC_stds = pd.read_csv('/home/azureuser/mycontainer/HBN-Processed/EOEC_std.csv')
print(EOEC_means.shape, EOEC_stds.shape)

In [ ]:
for i in range(1, 21):
    subject = EOEC_means.columns[i]

    EC_raw = mne.read_epochs(f'/home/azureuser/mycontainer/HBN-Processed/{subject}/epochs/EC.fif', preload=True, verbose=False)
    EO_raw = mne.read_epochs(f'/home/azureuser/mycontainer/HBN-Processed/{subject}/epochs/EO.fif', preload=True, verbose=False)

    EC_data = EC_raw.get_data()
    EO_data = EO_raw.get_data()

    print(f'{subject} EC: {EC_data.shape} EO: {EO_data.shape}, type: {type(EC_data)}')  

    # Subtract mean of each channel from each epoch and divide by standard deviation
    EC_data = (EC_data - EOEC_means[subject].to_numpy()[:, None]) / EOEC_stds[subject].to_numpy()[:, None]
    EO_data = (EO_data - EOEC_means[subject].to_numpy()[:, None]) / EOEC_stds[subject].to_numpy()[:, None]

    # Replace all NaNs with 0
    EC_data = np.nan_to_num(EC_data)
    EO_data = np.nan_to_num(EO_data)

    path = os.path.join("/home/azureuser/mycontainer/SmallSubset", subject)

    Path(path).mkdir(parents=True, exist_ok=True)

    torch.save(torch.tensor(EC_data), os.path.join(path, "EC.pt"))
    torch.save(torch.tensor(EO_data), os.path.join(path, "EO.pt"))

In [6]:
EOEC = pd.read_csv('/home/azureuser/mycontainer/HBN-Processed/EOEC_means.csv')
missing_patients = pd.read_csv('/home/azureuser/mycontainer/HBN-Processed/missing_patient_means.csv')
agg_patients = pd.read_csv('/home/azureuser/mycontainer/HBN-Processed/agg_mean.csv')

print(EOEC.shape, missing_patients.shape, agg_patients.shape)

(19, 2952) (19, 506) (19, 1695)


In [4]:
subfolders = set([f.path for f in os.scandir('/home/azureuser/mycontainer/HBN-Processed') if f.is_dir()])
print(len(subfolders))

3073


In [7]:
all_patients = set(EOEC.columns[1:]).union(set(missing_patients.columns[1:])).union(set(agg_patients.columns[1:]))
print(len(all_patients))

3080


In [9]:
print(len(set(EOEC.columns[1:])))
print(len(set(missing_patients.columns[1:])))
print(len(set(agg_patients.columns[1:])))

2951
505
1694


In [17]:
unix = set(EOEC.columns[1:]).intersection(set(missing_patients.columns[1:]))
print(len(unix))
print(unix)

479
{'NDARJA166LHK', 'NDARHD507TEF', 'NDARPP622WV4', 'NDAREH074NG8', 'NDARDD854GF8', 'NDARJX939UCQ', 'NDARBU757VJK', 'NDARAE866UVF', 'NDARAG340ERT', 'NDARNF873FCV', 'NDARMR039JV5', 'NDARJG298YVA', 'NDARPA827UBJ', 'NDARGD425PTN', 'NDARJH065GKG', 'NDARPD029JVJ', 'NDARHB764VZ2', 'NDARJZ274PRQ', 'NDARDZ528DLZ', 'NDARFA815FXE', 'NDARGF192VD1', 'NDARTK682LJE', 'NDARGL936KTV', 'NDARGE946RCB', 'NDARAB793GL3', 'NDARTF776EYR', 'NDARTK950DY5', 'NDARNY023FWG', 'NDARFF070XHV', 'NDARHF351AA6', 'NDARKU649DB7', 'NDARAC350XUM', 'NDARLM313FVX', 'NDARHR140GMB', 'NDARGA048BD8', 'NDARRH344TL7', 'NDARED628HMQ', 'NDARRW951BYF', 'NDARRX800KW8', 'NDARJP444DZM', 'NDARAC462DZH', 'NDARGG205WVN', 'NDARAN160GUF', 'NDARCV606ZZ5', 'NDARMH231UEP', 'NDARFM229HHA', 'NDARKP815KPZ', 'NDARCL105LUE', 'NDARJF027UGB', 'NDARKX665ZD3', 'NDARFG757CEU', 'NDARDT499DWP', 'NDARBC240LNC', 'NDARTC865YNZ', 'NDARCL016NHB', 'NDARBM642JFT', 'NDAREC542MH3', 'NDARLY687YDQ', 'NDARNE406XC3', 'NDARBL242L4H', 'NDARLL914UW4', 'NDARRU729XRU', 'ND

In [12]:
uni = set(missing_patients.columns[1:]).intersection(set(agg_patients.columns[1:]))
print(len(uni))

0


In [16]:
unix = set(EOEC.columns[1:]).intersection(set(agg_patients.columns[1:]))
print(len(unix))

1591


In [22]:
# Iterate through each directory and check if its empty
empty_folders = []
subfolders = [f.path for f in os.scandir('/home/azureuser/mycontainer/TUH-Processed') if f.is_dir()]
for folder in tqdm(subfolders):
    if not os.listdir(os.path.join(folder, 'epochs')):
        empty_folders.append(folder)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14987/14987 [01:27<00:00, 170.49it/s]


In [24]:
print(len(empty_folders))
print(empty_folders[0])

48
/home/azureuser/mycontainer/TUH-Processed/aaaaaarm


In [27]:
# Delete empty folders
for folder in tqdm(empty_folders):
    os.rmdir(folder)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 72.32it/s]
